In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Reproduksi, Analisis Penyebab, Perbaikan, & Uji Ulang**

In [3]:
import sqlite3
import pandas as pd

# Pastikan folder bukti Day 34 tersedia
os.makedirs("reports/evidence", exist_ok=True)

# 1. Cek isi docs/bug_list.csv sebelum diperbarui ke format Day 34
print("=== 1. ISI docs/bug_list.csv SAAT INI ===")
if os.path.exists("docs/bug_list.csv"):
    print(pd.read_csv("docs/bug_list.csv").to_string(index=False))
else:
    print("File docs/bug_list.csv belum ada.")

=== 1. ISI docs/bug_list.csv SAAT INI ===
bug_id tc_id                               modul tingkat_keparahan                                                                                                                                                                                       deskripsi_masalah                                                                                                  langkah_reproduksi status_perbaikan                     bukti
BUG-01 TC-05        Impor Data (src/importer.py)            Tinggi                  Baris komentar dengan tanggal tidak valid (2026-99-99 25:61:00) dan di luar rentang (1998-05-12 08:00:00) lolos diimpor (Diterima: 2) tanpa penolakan aturan validasi tanggal Q05/Q06.                   1. Buka menu Impor -> Comments. 2. Unggah tc05_tanggal_invalid.csv. 3. Klik Mulai Impor Comments.          Terbuka  reports/evidence/T05.png
BUG-02 TC-07    Eksplorasi Data (src/queries.py)            Tinggi Kombinasi filter rentang tanggal bersam

In [5]:
# 2. Verifikasi otomatis perbaikan kode untuk BUG-02, BUG-01, dan BUG-03
print("\n=== 2. VERIFIKASI UJI ULANG OTOMATIS (RE-TEST) ===")

# Test BUG-02 (filter_comments dengan rentang tanggal)
from src.queries import filter_comments
conn = sqlite3.connect("data/sentiment.db")
try:
    res_bug02 = filter_comments(conn, platform="Instagram", date_from="2026-09-01", date_to="2026-09-30")
    print(f"RE-TEST BUG-02 (Filter Tanggal): LULUS tanpa TypeError (hasil: {len(res_bug02)} baris).")
except Exception as e:
    print(f"RE-TEST BUG-02 GAGAL: {e}")

# Test BUG-01 (import_comments dengan tanggal invalid & luar rentang)
from src.importer import import_comments
conn_mem = sqlite3.connect(":memory:")
conn_mem.execute("CREATE TABLE posts (post_id TEXT PRIMARY KEY, platform TEXT, published_at TEXT, text_content TEXT)")
conn_mem.execute("INSERT INTO posts VALUES ('P_TEST', 'Instagram', '2026-09-01 10:00:00', 'Post uji')")
conn_mem.execute("""CREATE TABLE comments (
    comment_id TEXT PRIMARY KEY, post_id TEXT, text_content TEXT, text_original TEXT,
    text_hash TEXT, published_at TEXT, status_eksklusi TEXT, alasan_eksklusi TEXT, split TEXT
)""")
df_test_bug01 = pd.DataFrame([
    {"comment_id": "C_INV_1", "post_id": "P_TEST", "text_content": "Tes", "published_at": "2026-99-99 25:61:00"},
    {"comment_id": "C_INV_2", "post_id": "P_TEST", "text_content": "Tes", "published_at": "1998-05-12 08:00:00"}
])
res_bug01 = import_comments(conn_mem, df_test_bug01)
print(f"RE-TEST BUG-01 (Validasi Tanggal Impor): diterima={res_bug01['diterima']}, ditolak={res_bug01['ditolak']}")
conn_mem.close()
conn.close()


=== 2. VERIFIKASI UJI ULANG OTOMATIS (RE-TEST) ===
RE-TEST BUG-02 (Filter Tanggal): LULUS tanpa TypeError (hasil: 0 baris).
RE-TEST BUG-01 (Validasi Tanggal Impor): diterima=0, ditolak=[('C_INV_1', 'table comments has no column named commented_at'), ('C_INV_2', 'table comments has no column named commented_at')]


**Perbarui docs/bug_list.csv**

In [6]:
import pandas as pd

bug_rows = [
    {
        "bug_id": "BUG-02",
        "fitur": "Eksplorasi Data (src/queries.py)",
        "dampak": "Tinggi",
        "deskripsi": "Kombinasi filter rentang tanggal bersama platform Instagram memicu TypeError: Invalid comparison between dtype=datetime64[s] and date saat kolom commented_at kosong/NaT.",
        "langkah": "1. Buka menu Data. 2. Pilih Platform Instagram dan isi Dari Tanggal (2026/09/01) serta Sampai Tanggal (2026/09/30).",
        "penyebab": "Perbandingan langsung objek datetime64 hasil pd.to_datetime().dt.date yang mengandung NaT dengan objek datetime.date.",
        "perbaikan": "Normalisasi kedua sisi perbandingan ke pd.Timestamp (.dt.normalize()) dan tambahkan guard parsed_dates.notna() pada src/queries.py.",
        "status": "Diperbaiki",
        "tanggal_uji_ulang": "2026-11-23"
    },
    {
        "bug_id": "BUG-01",
        "fitur": "Impor Data (src/importer.py)",
        "dampak": "Tinggi",
        "deskripsi": "Baris komentar dengan tanggal tidak valid (2026-99-99 25:61:00) dan di luar rentang cakupan (1998-05-12 08:00:00) lolos diimpor tanpa penolakan aturan Q05/Q06.",
        "langkah": "1. Buka menu Impor -> Comments. 2. Unggah tc05_tanggal_invalid.csv. 3. Klik Mulai Impor Comments.",
        "penyebab": "Fungsi import_comments di src/importer.py belum memiliki validasi format tanggal (Q05) maupun batas rentang tahun cakupan proyek (Q06).",
        "perbaikan": "Tambahkan fungsi _validate_timestamp() di src/importer.py untuk menolak tanggal invalid (Q05) dan di luar rentang 2020-2026 (Q06) sebelum INSERT.",
        "status": "Diperbaiki",
        "tanggal_uji_ulang": "2026-11-23"
    },
    {
        "bug_id": "BUG-03",
        "fitur": "Ekspor Data (app/pages/6_Ekspor.py)",
        "dampak": "Sedang",
        "deskripsi": "Halaman Ekspor hanya mengunduh berkas CSV tunggal tanpa menyertakan file manifest/metadata ekspor (versi data, versi model, filter aktif, dan jumlah baris).",
        "langkah": "1. Buka menu Ekspor. 2. Klik Unduh Berkas CSV (Filtered Export). 3. Periksa berkas terunduh dan direktori proyek.",
        "penyebab": "Halaman app/pages/6_Ekspor.py baru menyediakan satu tombol unduh CSV dan belum menyusun objek metadata JSON pendamping.",
        "perbaikan": "Tambahkan penyusunan objek manifest JSON otomatis, pratinjau di layar, penyimpanan ke exports/manifest_export_latest.json, dan tombol unduh JSON di app/pages/6_Ekspor.py.",
        "status": "Diperbaiki",
        "tanggal_uji_ulang": "2026-11-23"
    }
]

df_bugs_d34 = pd.DataFrame(bug_rows)
df_bugs_d34.to_csv("docs/bug_list.csv", index=False, encoding="utf-8")

print("docs/bug_list.csv berhasil diperbarui & diurutkan (Tinggi -> Sedang):")
print(df_bugs_d34[["bug_id", "fitur", "dampak", "status", "tanggal_uji_ulang"]].to_string(index=False))

docs/bug_list.csv berhasil diperbarui & diurutkan (Tinggi -> Sedang):
bug_id                               fitur dampak     status tanggal_uji_ulang
BUG-02    Eksplorasi Data (src/queries.py) Tinggi Diperbaiki        2026-11-23
BUG-01        Impor Data (src/importer.py) Tinggi Diperbaiki        2026-11-23
BUG-03 Ekspor Data (app/pages/6_Ekspor.py) Sedang Diperbaiki        2026-11-23
